# Assignment 3 - Manyu Ghildiyal

## Import Statements

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.7.1+cu118
True
NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
from torch_hybrid import TorchLogisticMFScorer, ItemTags
from lenskit.data import Dataset, ItemListCollection
from lenskit.pipeline import topn_pipeline
from lenskit.knn import ItemKNNScorer, SLIMScorer
from lenskit.als import ImplicitMFScorer
from lenskit.basic import PopScorer
from lenskit.metrics import MeasurementCollector, RBP
from lenskit.batch import recommend

## Loading Data

In [3]:
#Loading a zip file using lenksit
loaded_works = Dataset.load("a3-train-data-micro/a3-train-data-micro")
loaded_parquet = ItemListCollection.load_parquet(
    "a3-tune-actions.parquet",
    key=['user_id'],
    layout='flat'
)

## New Method

In [4]:
scorer = TorchLogisticMFScorer()
pipe = topn_pipeline(scorer)
pipe.train(loaded_works)

In [5]:
test_users = loaded_parquet.keys()
new_test_users = [k.user_id for k in loaded_parquet.keys()]

In [6]:
recs = recommend(pipe, new_test_users, n=50)
recs

<ItemListCollection (10000 lists) {
  storage: 'list'
  key: 'user_id'
}>

### Analysis:
The hybrid model was built by combining two provided starter files: torchlmf.py (which handles implicit feedback and logistic/BPR loss functions) and torchtag.py (which handles sparse item feature embeddings). The key work was merging these two approaches into a single model that supports three content feature sources — authors, genres, and subject headings — while training on implicit feedback using either logistic or BPR loss.

Several decisions needed to be made during implementation. The most significant was how to handle the positive/negative item pairs during training versus single items during inference. During training, items come in pairs (one positive, one negative), so content embeddings needed to be computed separately for each and stacked to match the (B, 2, e_size) shape. During inference, only a single item is scored at a time, so stacking is skipped. This was handled with a conditional check on the item tensor's dimensions.
For the ablation study, use_authors, use_genres, and use_subjects flags were added to the config to allow selectively disabling content features without changing the model architecture.

My initial implementation was fairly brute force — I cared little about efficiency and just focused on making it run. Because of this, even a simple pipeline took nearly an hour to run on an Intel i9-10980HK CPU with 32GB of RAM and an NVIDIA RTX 3070 (8GB). While technically submittable, I wanted more efficient code for my final submission. The key fix was ensuring PyTorch was using the GPU rather than the CPU, which required reinstalling PyTorch with CUDA support (torch 2.7.1+cu118). After this change training speed improved dramatically.

## Setting Up Pipelines

### PopScorer

In [8]:
pop_scorer = PopScorer()
pop_pipe = topn_pipeline(pop_scorer)
pop_pipe.train(loaded_works)
pop_recs = recommend(pop_pipe, new_test_users, n=50)

C:\Users\MANYU\miniconda3\Lib\site-packages\lenskit\data\_relationships.py:930: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NaT' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stats.loc[stats["count"] == 0, "first_time"] = pd.NaT
C:\Users\MANYU\miniconda3\Lib\site-packages\lenskit\data\_relationships.py:932: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NaT' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stats.loc[stats["count"] == 0, "last_time"] = pd.NaT


### ItemKNNScorer - Removed due to Time Constraints

In [9]:
# knn_scorer = ItemKNNScorer(feedback='implicit')
# knn_pipe = topn_pipeline(knn_scorer)
# knn_pipe.train(loaded_works)
# knn_recs = recommend(knn_pipe, new_test_users, n=50)
# # Removed due to time constraints and re-runs. It score 0.022628

### SLIMScorer

In [10]:
slim_scorer = SLIMScorer(max_nbrs=250)
slim_pipe = topn_pipeline(slim_scorer)
slim_pipe.train(loaded_works)
slim_recs = recommend(slim_pipe, new_test_users, n=50)

### ImplicitMFScorer

In [11]:
implicit_scorer = ImplicitMFScorer(features=64)
implicit_pipe = topn_pipeline(implicit_scorer)
implicit_pipe.train(loaded_works)
implicit_recs = recommend(implicit_pipe, new_test_users, n=50)

## Evaluation

In [12]:
collect = MeasurementCollector()
collect.add_metric(RBP())

collect.add_collection_measurements(recs, loaded_parquet, algorithm="torch_hybrid")
collect.add_collection_measurements(pop_recs, loaded_parquet, algorithm="pop")
# collect.add_collection_measurements(knn_recs, loaded_parquet, algorithm="knn")
collect.add_collection_measurements(slim_recs, loaded_parquet, algorithm="slim")
collect.add_collection_measurements(implicit_recs, loaded_parquet, algorithm="imf")

In [13]:
collect.list_metrics().groupby('algorithm')['RBP'].mean().sort_values(ascending=False)

algorithm
slim            0.031294
imf             0.024191
torch_hybrid    0.017078
pop             0.011272
Name: RBP, dtype: float64

## Hyperparameter Tuning

### Regularization Strength

In [14]:
hyp_configs = [
    {"epochs": 1, "reg": 0.1},
    {"epochs": 2, "reg": 0.01},
    {"epochs": 3, "reg": 0.001},
    {"epochs": 4, "reg": 0.01},
    {"epochs": 5, "reg": 0.001},
]

collect2 = MeasurementCollector()
collect2.add_metric(RBP())

In [15]:
i = 1
for config in hyp_configs:
    hyp_implicit_scorer = ImplicitMFScorer(features=64, **config)
    hyp_implicit_pipe = topn_pipeline(hyp_implicit_scorer)
    hyp_implicit_pipe.train(loaded_works)
    hyp_implicit_recs = recommend(hyp_implicit_pipe, new_test_users, n=50)
    collect2.add_collection_measurements(hyp_implicit_recs, loaded_parquet, algorithm=f"hyp_implicit_recs{i}")
    i+=1

In [16]:
i = 1
for config in hyp_configs:
    hyp_scorer = TorchLogisticMFScorer(**config)
    hyp_pipe = topn_pipeline(hyp_scorer)
    hyp_pipe.train(loaded_works)
    hyp_recs = recommend(hyp_pipe, new_test_users, n=50)
    collect2.add_collection_measurements(hyp_recs, loaded_parquet, algorithm=f"torch_hybrid_tuned{i}")
    i+=1

In [17]:
collect2.list_metrics().groupby('algorithm')['RBP'].mean().sort_values(ascending=False)

algorithm
hyp_implicit_recs4     0.023979
hyp_implicit_recs5     0.023974
hyp_implicit_recs3     0.023257
hyp_implicit_recs2     0.022992
hyp_implicit_recs1     0.019886
torch_hybrid_tuned5    0.017638
torch_hybrid_tuned4    0.016947
torch_hybrid_tuned3    0.016722
torch_hybrid_tuned2    0.015957
torch_hybrid_tuned1    0.013368
Name: RBP, dtype: float64

### Results + Analysis

Hyperparameter tuning was approached with compute time as the primary constraint. Configurations were selected by varying epochs (1-5) and regularization strength (0.001, 0.01) since these are the parameters most likely to impact performance without requiring extensive search. The results of these runs are shown below.

Based on the results, epochs=5, reg=0.001 performed best and was selected for the final evaluation - despite it being only 0.00056 better. In hindsight, a more systematic approach such as grid search over embedding sizes would have been preferable, but was not feasible given compute constraints.

## Final Evaluation 

In [18]:
final_loaded_works = Dataset.load('a3-final-train-data')

In [19]:
final_loaded_parquet = ItemListCollection.load_parquet(
    "a3-eval-actions.parquet",
    key=['user_id'],
    layout='flat'
)

In [20]:
final_scorer = TorchLogisticMFScorer(epochs=5, reg=0.001)
final_pipe = topn_pipeline(final_scorer)
final_pipe.train(final_loaded_works)

In [21]:
final_new_test_users = [k.user_id for k in final_loaded_parquet.keys()]

In [22]:
final_recs = recommend(final_pipe, final_new_test_users, n=50)

In [23]:
collect.add_collection_measurements(final_recs, final_loaded_parquet, algorithm="FINAL TORCH")
collect.list_metrics().groupby('algorithm')['RBP'].mean().sort_values(ascending=False)

algorithm
slim            0.031294
imf             0.024191
FINAL TORCH     0.020258
torch_hybrid    0.017078
pop             0.011272
Name: RBP, dtype: float64

### Results + Analysis
For the final evaluation, the best hyperparameters from tuning were used: epochs=5, reg=0.001. The results show that SLIM performed best overall with an RBP of 0.031, followed by ImplicitMF at 0.024. The final tuned hybrid model scored 0.020, outperforming the standard hybrid (0.017) and PopScorer (0.011). The improvement from the standard to final hybrid — 0.017 to 0.020 — is meaningful in the context of recommender systems where small gains are significant.While the hybrid model did not outperform SLIM or ImplicitMF, this is expected as noted in the assignment. Both are strong baselines that are difficult to beat. The hybrid model does demonstrate clear improvement over PopScorer, confirming that personalization is valuable, and outperforms the untuned version, confirming that hyperparameter tuning had a positive effect.

## Ablation Study

In [29]:
ablation_configs = [
    {"name": "pure_collab",      "use_authors": False, "use_genres": False, "use_subjects": False},
    {"name": "collab_genres",    "use_authors": False, "use_genres": True,  "use_subjects": False},
    {"name": "collab_authors",   "use_authors": True,  "use_genres": False, "use_subjects": False},
    {"name": "collab_subjects",  "use_authors": False, "use_genres": False, "use_subjects": True},
    {"name": "full_model",       "use_authors": True,  "use_genres": True,  "use_subjects": True},
]

In [30]:
collect3 = MeasurementCollector()
collect3.add_metric(RBP())

In [31]:
best_config = {"epochs": 5, "reg": 0.001}
for ablation in ablation_configs:
    name = ablation.pop("name")
    ablation_scorer = TorchLogisticMFScorer(**best_config, **ablation)
    ablation_pipe = topn_pipeline(ablation_scorer)
    ablation_pipe.train(final_loaded_works)
    ablation_recs = recommend(ablation_pipe, final_new_test_users, n=50)
    collect3.add_collection_measurements(ablation_recs, final_loaded_parquet, algorithm=name)

In [32]:
collect3.list_metrics().groupby('algorithm')['RBP'].mean().sort_values(ascending=False)

algorithm
collab_authors     0.021277
full_model         0.020414
pure_collab        0.019882
collab_subjects    0.019670
collab_genres      0.019171
Name: RBP, dtype: float64

### Results + Analysis
The ablation study evaluated five versions of the hybrid model with different content features enabled or disabled.

Interestingly, collab_authors performed best at 0.021, even outperforming the full_model at 0.020. This suggests that author embeddings are the most valuable content feature in this model.

Looking at the results more closely, models without genres consistently performed better — collab_authors (0.021), pure_collab (0.020), and collab_subjects (0.020) all outperformed collab_genres (0.019). This suggests that genre embeddings may actually be hurting performance rather than helping, possibly because genres are too broad to provide useful personalization signal.

Author embeddings appear to be the most important content feature — they appear in the two best performing models (collab_authors and full_model). Subject headings show middling results, with collab_subjects performing slightly below pure_collab, suggesting subjects provide minimal benefit on their own.

Based on these results, an optimized version of this model would likely use authors only or authors combined with subjects, dropping genres entirely. This finding is somewhat counterintuitive since genres are commonly used in recommender systems, but may reflect the specific characteristics of this GoodReads dataset or some issue in the implementation.

## Reflection

This project taught me several valuable lessons about building and evaluating recommender systems in practice.

The most technically challenging part was combining torchlmf.py and torchtag.py into a single hybrid model. Debugging was particularly painful because every test required retraining the model, which took nearly an hour on CPU. A single shape mismatch error meant waiting an hour just to see the next error message. This forced me to think more carefully before running code rather than relying on trial and error.

I also underestimated the importance of hyperparameter tuning going in. I treated it as a box to check rather than a meaningful part of the process, but the results showed a clear and measurable improvement between the default and tuned configurations. In future projects I would invest more time here.

Perhaps the most practical lesson was hardware awareness. I initially did not realize PyTorch was using my CPU instead of my GPU, which was the primary cause of the slow training times. Once I reinstalled PyTorch with CUDA support and confirmed my RTX 3070 was being used, performance improved dramatically. Always verify your compute environment before starting long training runs.

Finally, this project reinforced the importance of pacing. Attempting to complete everything in one day would have been impossible given the training times involved. Planning around compute constraints is as important as the implementation itself.